# **Example notebook: How to use the OS-NS SAR oil slick detection model** <br>


**Notebook structure**
1. Use model on tiles and save predictions
2. Use model on labeled dataset - return segmentation metrics (Dice/IoU)
3. Use model on large geotif - tile and mosaic predictions as a result

*Model info*
---

**Input data** <br>
SAR co-polarization (VV or HH) intensity

**Preprocessing required** 
* [Optional - Recommended] Project SAR VV/HH images
* [Optional] Radiometric corrections and despeckling
* [Optional] Land mask. (Use 0 as nodata value)
* Tile so that each tile covers approximately 10 km (10.24 km used during training).<br>  Recommended tiling setup:
    * Tile size: 512 px
    * Pixel size: 20 m  

------

# Use model on tiles and save predictions

**Organize dataset in this format**
* dataset_folder/
    * images/
        * <whatever_tile_name_01>.tif
        * <whatever_tile_name_02>.tif
        * ...

**Predictions will be saved at** 
* dataset_folder/
    * preds/
        * <whatever_tile_name_01>.tif
        * <whatever_tile_name_02>.tif
        * ...


In [ ]:
import os
from glob import glob
import segmentation_models_pytorch as smp
from my_code.inference import predict_pytorch
from my_code.utils.transforms import sar_transform

model = smp.from_pretrained('NS-OS_512_20_model_v1.pth')
transform = sar_transform(512, triple=True)

tile_list = glob(os.path.join('dataset/images', '*.tif'))
preds_folder = os.path.join('dataset/preds')

predict_pytorch(
    model=model,
    tile_paths=tile_list,
    transform=transform,
    predict_prob_or_class='class',
    save_out_preds=True,
    out_pred_path=preds_folder,
    batch_size=8
)

# Use model on labeled dataset - return segmentation metrics (Dice/IoU)

**Organize dataset in this format**
* dataset_folder/
    * images/
        * <whatever_tile_name_01>.tif
        * <whatever_tile_name_02>.tif
        * ...
    * labels/
        * <whatever_tile_name_01>.tif
        * <whatever_tile_name_02>.tif
        * ...

In [ ]:
import os
from glob import glob
import segmentation_models_pytorch as smp
from my_code.inference import predict_pytorch
from my_code.utils.transforms import sar_transform

model = smp.from_pretrained('NS-OS_512_20_model_v1.pth')
transform = sar_transform(512, triple=True)

tile_list = glob(os.path.join('dataset/images', '*.tif'))

dice, iou = predict_pytorch(
    model=model,
    tile_paths=tile_list,
    transform=transform,
    predict_prob_or_class='class',
    return_dice_iou=True,
    save_out_preds=False,
    batch_size=8
)

print(f'Dataset metrics \n Dice score: {dice} \n IoU score: {iou}')

# Use model on large geotif - tile and mosaic predictions as a result

**Organize dataset in this format**
* dataset_folder/
    * image.tif
        
**Mosaicked prediction will be saved at** 
* dataset_folder/
    * mosaic_pred.tif

**And, optionally, vectorized in a shapefile at**
* dataset_folder/
    * mosaic_pred.shp

In [ ]:
import os
from glob import glob
import segmentation_models_pytorch as smp
from my_code.inference_mosaic import inference_mosaic
from my_code.utils.transforms import sar_transform

TIF_IMG_PATH = '' # define geotif img path
#NOTE: predictions will be saved in the same dir as TIF_IMG_PATH.

model = smp.from_pretrained('NS-OS_512_20_model_v1.pth')
transform = sar_transform(512, triple=True)

inference_mosaic(
    tif_img_path=TIF_IMG_PATH,
    model=model,
    transform=transform,
    landmask_path=None, #specify binary landmask path as needed
    predict_prob_or_class='class',
    mosaic_tiles=True,
    padding='auto',
    mosaic_classify_softmax=True,
    vectorize_output=True,
    vector_format='shp'
)